# Práctica de la lezione 11: clustering con Mean Shift

En esta pratica aplicas lo que viste en lezione sobre Mean Shift y el papel del bandwidth. Encuentras dos esercizios. Cada uno trae un consegna con suggerimenti, una o varias cellas de código con partes marcadas como `# TODO` para que completes, y más abajo una soluzione colapsada por si te atascas. Intenta resolverlo da solo antes de mirarla.

En el esercizio uno vuelves al Titanic y exploras cómo cambia el número de grupos que encuentra Mean Shift según el `quantile` que le pases a `estimate_bandwidth`. En el esercizio dos programas tu propia funzione de "un paso" de Mean Shift sobre un dataset pequeño distinto al de lezione, generado con `make_blobs`.

## Ejercicio 1: prueba distintos valorees de `quantile` sobre el Titanic

Tienes ya el Titanic cargado y preparado exactamente como en lezione: huecos rellenos, colonnas de texto convertidas en números y dati escalados con `StandardScaler`. Tu tarea es completar un bucle que recorra una lista de valorees de `quantile`, calcule el bandwidth estimado con `estimate_bandwidth` para cada uno, ajuste un `MeanShift` con ese bandwidth y guarde el número de grupos encontrado.

**Suggerimenti:**

- Usa `estimate_bandwidth(X_escalado, quantile=q, random_state=42)` para que el risultato sea reproducible
- Crea el modello con `MeanShift(bandwidth=...)` y entrénalo con `.fit(X_escalado)`, igual que en lezione
- El número de grupos es `len(np.unique(modello.labels_))`
- Recuerda lo que viste en lezione: cuanto más alto es `quantile`, mayor tiende a salir el bandwidth, y menos grupos encontrarás

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MeanShift, estimate_bandwidth

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)

titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
titanic['Fare'] = titanic['Fare'].fillna(titanic['Fare'].median())
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])
titanic['Sex'] = titanic['Sex'].map({'male': 0, 'female': 1})
titanic['Embarked'] = titanic['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
titanic.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

X = titanic.drop('Survived', axis=1)
X_escalado = StandardScaler().fit_transform(X)

print(f"Filas: {X_escalado.shape[0]}, columnas: {X_escalado.shape[1]}")

In [ ]:
valores_de_quantile = [0.05, 0.1, 0.2, 0.3, 0.5]
resultados = []

for q in valores_de_quantile:
    # TODO 1: calcula el bandwidth estimado para este quantile, con
    #         estimate_bandwidth(X_escalado, quantile=q, random_state=42)
    bandwidth = ...

    # TODO 2: crea un modelo MeanShift con ese bandwidth y entrénalo con X_escalado
    modelo = ...

    # TODO 3: calcula el número de grupos encontrados y añádelo a la lista resultados
    n_grupos = ...
    resultados.append(n_grupos)

    print(f"quantile={q} -> bandwidth={bandwidth:.2f} -> {n_grupos} grupos")

Ejecuta la cella anterior. Si las variabiles están bien completadas, deberías ver que el número de grupos baja de forma bastante marcada a medida que `quantile` sube: con `quantile=0.05` salen decenas de grupos pequeños, y con `quantile=0.5` el risultato se acerca mucho a los tres grupos grandes que también encuentra K-Means cuando lo fuerzas a `n_clusters=3`.

<details>
<summary>💡 Vedi soluzione</summary>

```python
valorees_de_quantile = [0.05, 0.1, 0.2, 0.3, 0.5]
risultatos = []

for q in valorees_de_quantile:
    # TODO 1: bandwidth estimado para este quantile
    bandwidth = estimate_bandwidth(X_escalado, quantile=q, random_state=42)

    # TODO 2: modello MeanShift entrenado con ese bandwidth
    modello = MeanShift(bandwidth=bandwidth)
    modello.fit(X_escalado)

    # TODO 3: número de grupos encontrados
    n_grupos = len(np.unique(modello.labels_))
    risultatos.append(n_grupos)

    print(f"quantile={q} -> bandwidth={bandwidth:.2f} -> {n_grupos} grupos")
```

Con este dataset, el patrón es claro: `quantile=0.05` produce bandwidths pequeños y decenas de grupos, muchos de ellos con un puñado de pasajeros; a medida que subes `quantile`, el bandwidth estimado crece y el número de grupos cae, hasta acercarse a tres con `quantile=0.5`, un risultato comparable al que obtienes forzando `KMeans` con `n_clusters=3`. La diferencia es que aquí llegaste a esa cifra explorando la densidad de los dati, no imponiéndola de antemano.

</details>

## Ejercicio 2: programa un paso de Mean Shift sobre un dataset nuevo

En lezione programaste `MeanShiftPropio`, que repite el desplazamiento de todos los centroides hasta la convergencia. Aquí vas a completar una versión mucho más simple: una funzione que da un único paso de Mean Shift para un solo punto, es decir, calcula hacia dónde se movería ese punto si mirara una sola vez a sus vecinos dentro de un radio dado.

Trabajas sobre un dataset distinto al de lezione, con cuatro nubes de puntos en lugar de tres.

**Suggerimenti:**

- Recorre todos los puntos de `dati` y, para cada uno, comprueba si su distancia a `punto` es menor que `radio`, con `np.linalg.norm(otro - punto) < radio`
- Guarda en una lista los puntos que cumplen esa condición
- El nuevo punto es la media de esa lista de vecinos, con `np.average(vecinos, axis=0)`
- Si aplicas la funzione varias veces seguidas sobre su propio risultato, el punto debería acercarse cada vez más al centro de la nube más cercana, hasta dejar de moverse

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Un dataset distinto al de clase: cuatro nubes de puntos
X_nuevo, _ = make_blobs(n_samples=40, centers=4, cluster_std=0.9, random_state=7)

plt.scatter(X_nuevo[:, 0], X_nuevo[:, 1], s=60, edgecolors='k')
plt.title("Dataset nuevo para el ejercicio 2")
plt.show()

In [ ]:
def paso_mean_shift(punto, datos, radio):
    vecinos = []

    for otro in datos:
        # TODO 1: si la distancia entre 'otro' y 'punto' es menor que 'radio',
        #         añade 'otro' a la lista de vecinos
        ...

    # TODO 2: calcula la media de los vecinos y devuélvela como el nuevo punto
    nuevo_punto = ...
    return nuevo_punto


# Partimos de uno de los puntos del dataset y damos varios pasos seguidos
punto_actual = X_nuevo[0].copy()
radio = 3

for paso in range(6):
    punto_siguiente = paso_mean_shift(punto_actual, X_nuevo, radio)
    desplazamiento = np.linalg.norm(punto_siguiente - punto_actual)
    print(f"Paso {paso}: posición {punto_siguiente}, desplazamiento {desplazamiento:.4f}")
    punto_actual = punto_siguiente

Ejecuta la cella anterior. Si `paso_mean_shift` está bien completada, deberías ver que el desplazamiento entre pasos se hace cada vez más pequeño, hasta llegar a cero: el punto ha "subido la colina de densidad" y se ha quedado quieto en el centro de su nube, exactamente igual que los centroides de `MeanShiftPropio` en lezione, solo que aquí lo ves paso a paso para un único punto en vez de para todos a la vez y con fusión de centroides.

<details>
<summary>💡 Vedi soluzione</summary>

```python
def paso_mean_shift(punto, dati, radio):
    vecinos = []

    for otro in dati:
        # TODO 1: nos quedamos con los puntos dentro del radio
        if np.linalg.norm(otro - punto) < radio:
            vecinos.append(otro)

    # TODO 2: el nuevo punto es la media de esos vecinos
    nuevo_punto = np.average(vecinos, axis=0)
    return nuevo_punto


punto_actual = X_nuevo[0].copy()
radio = 3

for paso in range(6):
    punto_siguiente = paso_mean_shift(punto_actual, X_nuevo, radio)
    desplazamiento = np.linalg.norm(punto_siguiente - punto_actual)
    print(f"Paso {paso}: posición {punto_siguiente}, desplazamiento {desplazamiento:.4f}")
    punto_actual = punto_siguiente
```

Con `radio=3`, el punto de partida suele necesitar solo dos o tres pasos para converger: el desplazamiento cae rápidamente hasta 0,0 y, a partir de ahí, `paso_mean_shift` devuelve siempre la misma posición porque el conjunto de vecinos dentro del radio ya no cambia de una llamada a la siguiente. Si repites el experimento partiendo de otro punto de `X_nuevo`, o cambias el valore de `radio`, comprueba si el punto converge a una nube distinta o si necesita más pasos para estabilizarse.

</details>

## Resumen: lo que te llevas de esta pratica

- El `quantile` de `estimate_bandwidth` controla, de forma indirecta, cuántos grupos encuentra Mean Shift: subirlo agranda el bandwidth estimado y reduce el número de grupos
- Con un `quantile` alto, Mean Shift puede llegar a un número de grupos comparable al que obtienes forzando K-Means con `n_clusters=3`, pero descubierto a partir de la densidad de los dati, no impuesto de antemano
- Un solo paso de Mean Shift consiste en promediar los vecinos de un punto dentro de un radio fijo; repetir ese paso una y otra vez es exactamente lo que hace converger a cada centroide hacia una cima de densidad
- El desplazamiento entre pasos consecutivos tiende a cero según el punto se acerca a esa cima, la misma condición de convergencia que usaste en `MeanShiftPropio` durante la lezione

**Abbreviazioni usate en esta pratica:** CSV (*comma-separated values*, valorees separados por comas).